In [3]:
import numpy as np
import meshio

msh      = meshio.read("wing_surface.xdmf")
pts      = msh.points
tris     = msh.cells_dict["triangle"]
normals  = msh.point_data["normals"]
p_gauge  = msh.point_data["p_gauge"]
wss      = msh.point_data["shearstress"]

v0    = pts[tris[:,0]]
v1    = pts[tris[:,1]]
v2    = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
n_center = (normals[tris[:,0]] + normals[tris[:,1]] + normals[tris[:,2]]) / 3.0

# ── Decompose: pressure vs viscous ──────────────────────────────
traction     = msh.point_data["traction"]
t_center     = (traction[tris[:,0]] + traction[tris[:,1]] + traction[tris[:,2]]) / 3.0

wss_center   = (wss[tris[:,0]] + wss[tris[:,1]] + wss[tris[:,2]]) / 3.0
p_center     = (p_gauge[tris[:,0]] + p_gauge[tris[:,1]] + p_gauge[tris[:,2]]) / 3.0
F_pressure   = np.sum(-p_center[:, np.newaxis] * n_center * areas[:, np.newaxis], axis=0)
F_viscous    = np.sum(wss_center * areas[:, np.newaxis], axis=0)
F_total      = F_pressure + F_viscous

print(f"F_pressure = {F_pressure}")
print(f"F_viscous  = {F_viscous}")
print(f"F_total    = {F_total}")

# ── Check normal integral (open surface effect) ──────────────────
int_n = np.sum(n_center * areas[:, np.newaxis], axis=0)
print(f"\nIntegral normals = {int_n}")
print(f"→ Open surface contribution to force = {-p_gauge.mean() * int_n}")

# ── Check: span direction ────────────────────────────────────────
print(f"\nWing oriented:")
print(f"  chord → x? range = {pts[:,0].min():.3f} → {pts[:,0].max():.3f}")
print(f"  span  → y? range = {pts[:,1].min():.3f} → {pts[:,1].max():.3f}")
print(f"  thick → z? range = {pts[:,2].min():.3f} → {pts[:,2].max():.3f}")

F_pressure = [  851.9576   -214.38313 17780.338  ]
F_viscous  = [-266.99197      -0.97438264   10.573612  ]
F_total    = [  584.9656   -215.35751 17790.912  ]

Integral normals = [ 0.00771111  0.07531793 -0.00055922]
→ Open surface contribution to force = [ 31.22092   304.94904    -2.2641943]

Wing oriented:
  chord → x? range = 0.000 → 2.491
  span  → y? range = 0.000 → 3.000
  thick → z? range = -0.078 → 0.044


In [5]:
import numpy as np
import meshio

msh      = meshio.read("wing_surface.xdmf")
pts      = msh.points
tris     = msh.cells_dict["triangle"]
normals  = msh.point_data["normals"]
p_raw    = msh.point_data["p_gauge"] +  54019.55  # restore p absolute
wss      = msh.point_data["shearstress"]

# Thử 3 cases
F_of = np.array([1133.0760, -4433.4000, 17824.7600])

v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)

def compute_force(p_test, normals, wss, tris, areas):
    traction = -p_test[:, np.newaxis] * normals + wss
    t_center = (traction[tris[:,0]] + traction[tris[:,1]] + traction[tris[:,2]]) / 3.0
    return np.sum(t_center * areas[:, np.newaxis], axis=0)

# Case 1: p absolute, pRef = 0 (OpenFOAM default)
F1 = compute_force(p_raw, normals, wss, tris, areas)

# Case 2: p gauge với p_inf = 54019.55
F2 = compute_force(p_raw - 54019.55, normals, wss, tris, areas)

# Case 3: không có wss
F3 = compute_force(p_raw - 54019.55, normals, np.zeros_like(wss), tris, areas)

print(f"{'':6} {'OpenFOAM':>12} {'pRef=0':>12} {'pRef=54019':>12} {'no wss':>12}")
print(f"{'-'*54}")
for i, label in enumerate(["Fx", "Fy", "Fz"]):
    print(f"{label:6} {F_of[i]:12.2f} {F1[i]:12.2f} {F2[i]:12.2f} {F3[i]:12.2f}")
print(f"{'|F|':6} {np.linalg.norm(F_of):12.2f} "
      f"{np.linalg.norm(F1):12.2f} "
      f"{np.linalg.norm(F2):12.2f} "
      f"{np.linalg.norm(F3):12.2f}")

# Bonus: check integral normals
n_center = (normals[tris[:,0]] + normals[tris[:,1]] + normals[tris[:,2]]) / 3.0
int_n    = np.sum(n_center * areas[:, np.newaxis], axis=0)
print(f"\nIntegral normals = {int_n}")
print(f"p_inf * int_n    = {54019.55 * int_n}")

           OpenFOAM       pRef=0   pRef=54019       no wss
------------------------------------------------------
Fx          1133.08       184.68       601.35       868.32
Fy         -4433.40     -4294.19      -225.54      -224.57
Fz         17824.76     17846.44     17816.04     17805.60
|F|        18402.74     18356.73     17827.61     17828.17

Integral normals = [ 0.00771111  0.07531793 -0.00055922]
p_inf * int_n    = [ 416.5506  4068.6406   -30.20896]


In [10]:
import numpy as np
import meshio
import pyvista as pv

# ── Load OpenFOAM surface ────────────────────────────────────────
msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]
p_inf   = 54019.55

# ── Tìm root boundary (y = 0) ────────────────────────────────────
y_min = pts[:,1].min()
print(f"Root at y = {y_min:.6f} m")

# Tìm các điểm tại root
root_mask = np.abs(pts[:,1] - y_min) < 1e-4
root_pts_idx = np.where(root_mask)[0]
root_pts = pts[root_pts_idx]
print(f"Root boundary points: {len(root_pts_idx)}")

# ── Tạo root cap bằng cách triangulate ──────────────────────────
# Project root points xuống 2D (x-z plane)
from scipy.spatial import Delaunay

root_2d = root_pts[:, [0, 2]]  # x-z coordinates
tri_2d  = Delaunay(root_2d)
root_tris_local = tri_2d.simplices  # triangles trong local index

# Convert sang global index
root_tris_global = root_pts_idx[root_tris_local]
print(f"Root cap triangles: {len(root_tris_global)}")

# ── Tính normal của root cap ─────────────────────────────────────
# Root cap normal = (0, -1, 0) hướng vào trong (y = 0, pointing -y)
root_normal = np.array([0.0, -1.0, 0.0])

# ── Tính diện tích root cap ──────────────────────────────────────
v0 = pts[root_tris_global[:,0]]
v1 = pts[root_tris_global[:,1]]
v2 = pts[root_tris_global[:,2]]
root_areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
total_root_area = root_areas.sum()
print(f"Root cap area: {total_root_area:.6f} m²")

# ── Pressure tại root cap = p_inf (freestream) ───────────────────
# Traction = -(p - p_inf) * n = 0 car p = p_inf à la racine
# Donc la contribution nette = 0 pour root cap avec p_gauge
# MAIS: on doit corriger l'integral des normales

# Correction: ajouter contribution root cap à l'intégrale
p_root = np.mean(p_raw[root_mask])  # pressure moyenne à la racine
print(f"Mean pressure at root: {p_root:.2f} Pa")
p_root_gauge = p_root - p_inf

# Force from root cap
F_root_cap = -(p_root_gauge) * root_normal * total_root_area
print(f"\nRoot cap force correction:")
print(f"  Fx = {F_root_cap[0]:.4f} N")
print(f"  Fy = {F_root_cap[1]:.4f} N")
print(f"  Fz = {F_root_cap[2]:.4f} N")

# ── Tính F_mapped từ wing surface ───────────────────────────────
p_gauge  = p_raw - p_inf
traction = -p_gauge[:, np.newaxis] * normals + wss

v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
t_center = (traction[tris[:,0]] + traction[tris[:,1]] + traction[tris[:,2]]) / 3.0
F_wing   = np.sum(t_center * areas[:, np.newaxis], axis=0)

# ── Total = wing + root cap ──────────────────────────────────────
F_total = F_wing + F_root_cap
F_of    = np.array([1133.0760, -4433.4000, 17824.7600])

print(f"\n=== Validation avec root cap correction ===")
print(f"{'':6} {'OpenFOAM':>12} {'Wing only':>12} {'+ Root cap':>12}")
print(f"{'-'*48}")
for i, label in enumerate(["Fx", "Fy", "Fz"]):
    print(f"{label:6} {F_of[i]:12.4f} {F_wing[i]:12.4f} {F_total[i]:12.4f}")
print(f"{'|F|':6} {np.linalg.norm(F_of):12.4f} "
      f"{np.linalg.norm(F_wing):12.4f} "
      f"{np.linalg.norm(F_total):12.4f}")

err_total = abs(np.linalg.norm(F_total) - np.linalg.norm(F_of)) / np.linalg.norm(F_of) * 100
print(f"\n|F| error = {err_total:.2f}%")

Root at y = 0.000000 m
Root boundary points: 627
Root cap triangles: 870
Root cap area: 0.080978 m²
Mean pressure at root: 51356.25 Pa

Root cap force correction:
  Fx = 0.0000 N
  Fy = -215.6690 N
  Fz = 0.0000 N

=== Validation avec root cap correction ===
           OpenFOAM    Wing only   + Root cap
------------------------------------------------
Fx        1133.0760     601.3506     601.3506
Fy       -4433.4000    -225.5436    -441.2126
Fz       17824.7600   17816.0352   17816.0352
|F|      18402.7434   17827.6074   17831.6404

|F| error = 3.10%


In [11]:
import numpy as np
import meshio
import pyvista as pv

# ── Load ─────────────────────────────────────────────────────────
msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]

# ── Check normal y-component distribution ────────────────────────
ny = normals[:, 1]
print(f"ny range: {ny.min():.4f} → {ny.max():.4f}")
print(f"ny > 0 (pointing +y): {(ny > 0).sum()} points ({(ny>0).mean()*100:.1f}%)")
print(f"ny < 0 (pointing -y): {(ny < 0).sum()} points ({(ny<0).mean()*100:.1f}%)")
print(f"ny ≈ 0 (chord dir)  : {(np.abs(ny) < 0.1).sum()} points")

# ── Visualize normal y-component ─────────────────────────────────
cloud = pv.PolyData(pts)
cloud["ny"]    = ny
cloud["p_raw"] = p_raw
cloud.save("check_normals.vtp")
print("\nSaved: check_normals.vtp")
print("Open in ParaView, color by 'ny'")
print("  Blue (ny < 0) = normals pointing toward root")  
print("  Red  (ny > 0) = normals pointing toward tip")
print("  → Phải thấy pattern rõ ràng upper/lower surface")

ny range: -1.0000 → 1.0000
ny > 0 (pointing +y): 86887 points (45.8%)
ny < 0 (pointing -y): 102668 points (54.2%)
ny ≈ 0 (chord dir)  : 132515 points

Saved: check_normals.vtp
Open in ParaView, color by 'ny'
  Blue (ny < 0) = normals pointing toward root
  Red  (ny > 0) = normals pointing toward tip
  → Phải thấy pattern rõ ràng upper/lower surface


In [13]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55  # restore absolute p
wss     = msh.point_data["shearstress"]
F_of    = np.array([1133.0760, -4433.4000, 17824.7600])

# ── Wing surface force (pRef=0, absolute pressure) ───────────────
traction = -p_raw[:, np.newaxis] * normals + wss

v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
t_center = (traction[tris[:,0]] + traction[tris[:,1]] + traction[tris[:,2]]) / 3.0
F_wing   = np.sum(t_center * areas[:, np.newaxis], axis=0)

# ── Root cap correction ───────────────────────────────────────────
y_min     = pts[:,1].min()
root_mask = np.abs(pts[:,1] - y_min) < 1e-4
root_idx  = np.where(root_mask)[0]
root_pts  = pts[root_idx]
root_tris = Delaunay(root_pts[:, [0,2]]).simplices

v0r = root_pts[root_tris[:,0]]
v1r = root_pts[root_tris[:,1]]
v2r = root_pts[root_tris[:,2]]
areas_r   = 0.5 * np.linalg.norm(np.cross(v1r-v0r, v2r-v0r), axis=1)
p_center_r = (p_raw[root_idx][root_tris[:,0]] +
              p_raw[root_idx][root_tris[:,1]] +
              p_raw[root_idx][root_tris[:,2]]) / 3.0
F_root = np.sum(-p_center_r[:, np.newaxis]
                * np.array([0., -1., 0.])
                * areas_r[:, np.newaxis], axis=0)

# ── Total ─────────────────────────────────────────────────────────
F_total = F_wing + F_root

print(f"=== Validation (pRef=0 + root cap) ===")
print(f"{'':6} {'OpenFOAM':>12} {'Wing':>12} {'Root cap':>12} {'Total':>12} {'Error%':>8}")
print(f"{'-'*58}")
for i, label in enumerate(["Fx", "Fy", "Fz"]):
    err = abs(F_total[i] - F_of[i]) / (abs(F_of[i]) + 1e-10) * 100
    print(f"{label:6} {F_of[i]:12.2f} {F_wing[i]:12.2f} "
          f"{F_root[i]:12.2f} {F_total[i]:12.2f} {err:8.2f}%")
err_mag = abs(np.linalg.norm(F_total) - np.linalg.norm(F_of)) / np.linalg.norm(F_of) * 100
print(f"{'|F|':6} {np.linalg.norm(F_of):12.2f} {np.linalg.norm(F_wing):12.2f} "
      f"{np.linalg.norm(F_root):12.2f} {np.linalg.norm(F_total):12.2f} {err_mag:8.2f}%")

=== Validation (pRef=0 + root cap) ===
           OpenFOAM         Wing     Root cap        Total   Error%
----------------------------------------------------------
Fx          1133.08       184.68         0.00       184.68    83.70%
Fy         -4433.40     -4294.19      4127.45      -166.74    96.24%
Fz         17824.76     17846.44         0.00     17846.44     0.12%
|F|        18402.74     18356.73      4127.45     17848.17     3.01%


In [15]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]
F_of    = np.array([1133.0760, -4433.4000, 17824.7600])

# ── Wing surface ──────────────────────────────────────────────────
traction = -p_raw[:, np.newaxis] * normals + wss
v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
t_center = (traction[tris[:,0]] + traction[tris[:,1]] + traction[tris[:,2]]) / 3.0
F_wing   = np.sum(t_center * areas[:, np.newaxis], axis=0)

# ── Tip cap (y = y_max) ───────────────────────────────────────────
y_max    = pts[:,1].max()
tip_mask = np.abs(pts[:,1] - y_max) < 1e-4
tip_idx  = np.where(tip_mask)[0]
tip_pts  = pts[tip_idx]
tip_tris = Delaunay(tip_pts[:, [0,2]]).simplices

print(f"Tip at y = {y_max:.4f} m")
print(f"Tip boundary points: {len(tip_idx)}")
print(f"Tip cap triangles  : {len(tip_tris)}")

v0t = tip_pts[tip_tris[:,0]]
v1t = tip_pts[tip_tris[:,1]]
v2t = tip_pts[tip_tris[:,2]]
areas_t    = 0.5 * np.linalg.norm(np.cross(v1t-v0t, v2t-v0t), axis=1)
p_center_t = (p_raw[tip_idx][tip_tris[:,0]] +
              p_raw[tip_idx][tip_tris[:,1]] +
              p_raw[tip_idx][tip_tris[:,2]]) / 3.0

print(f"Tip cap area: {areas_t.sum():.6f} m²")

# Test cả 2 normal directions
for sign, label in [(+1, "tip normal = +y"), (-1, "tip normal = -y")]:
    F_tip   = np.sum(-p_center_t[:, np.newaxis]
                     * np.array([0., float(sign), 0.])
                     * areas_t[:, np.newaxis], axis=0)
    F_total = F_wing + F_tip
    err_mag = abs(np.linalg.norm(F_total) - np.linalg.norm(F_of)) / np.linalg.norm(F_of) * 100

    print(f"\n=== {label} ===")
    for i, l in enumerate(["Fx", "Fy", "Fz"]):
        e = abs(F_total[i] - F_of[i]) / (abs(F_of[i]) + 1e-10) * 100
        print(f"{l}: OF={F_of[i]:10.2f}  mapped={F_total[i]:10.2f}  err={e:.1f}%")
    print(f"|F| error = {err_mag:.2f}%")

Tip at y = 3.0001 m
Tip boundary points: 1550
Tip cap triangles  : 2902
Tip cap area: 0.019593 m²

=== tip normal = +y ===
Fx: OF=   1133.08  mapped=    184.68  err=83.7%
Fy: OF=  -4433.40  mapped=  -5227.39  err=17.9%
Fz: OF=  17824.76  mapped=  17846.44  err=0.1%
|F| error = 1.06%

=== tip normal = -y ===
Fx: OF=   1133.08  mapped=    184.68  err=83.7%
Fy: OF=  -4433.40  mapped=  -3360.99  err=24.2%
Fz: OF=  17824.76  mapped=  17846.44  err=0.1%
|F| error = 1.31%


In [17]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]

v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)

def integrate(field):
    fc = (field[tris[:,0]] + field[tris[:,1]] + field[tris[:,2]]) / 3.0
    return np.sum(fc * areas[:, np.newaxis], axis=0)

# Decompose pressure vs viscous
F_p   = integrate(-p_raw[:, np.newaxis] * normals)
F_wss = integrate(wss)
F_tot = F_p + F_wss

# OpenFOAM reference
OF_p   = np.array([868.51,  -4434.20, 17835.28])
OF_wss = np.array([264.57,      0.80,    -10.51])
OF_tot = np.array([1133.08, -4433.40, 17824.76])

print(f"{'':12} {'OF_pressure':>12} {'mapped_p':>12} {'err%':>8}")
print(f"{'-'*48}")
for i, l in enumerate(["Fx", "Fy", "Fz"]):
    e = abs(F_p[i] - OF_p[i]) / (abs(OF_p[i]) + 1e-10) * 100
    print(f"{'pressure '+l:12} {OF_p[i]:12.2f} {F_p[i]:12.2f} {e:8.1f}%")

print()
print(f"{'':12} {'OF_viscous':>12} {'mapped_wss':>12} {'err%':>8}")
print(f"{'-'*48}")
for i, l in enumerate(["Fx", "Fy", "Fz"]):
    e = abs(F_wss[i] - OF_wss[i]) / (abs(OF_wss[i]) + 1e-10) * 100
    print(f"{'viscous '+l:12} {OF_wss[i]:12.2f} {F_wss[i]:12.2f} {e:8.1f}%")

# Check wss magnitude
wss_mag = np.linalg.norm(wss, axis=1)
print(f"\nwss magnitude: {wss_mag.min():.4f} → {wss_mag.max():.4f} Pa")
print(f"wss mean     : {wss_mag.mean():.4f} Pa")

# Check: wss direction vs normals
dot_wss_n = np.sum(wss * normals, axis=1)
print(f"\nDot(wss, normal): {dot_wss_n.min():.4f} → {dot_wss_n.max():.4f}")
print(f"(phải ≈ 0 vì wss tangential to surface)")

              OF_pressure     mapped_p     err%
------------------------------------------------
pressure Fx        868.51       451.73     48.0%
pressure Fy      -4434.20     -4293.16      3.2%
pressure Fz      17835.28     17836.00      0.0%

               OF_viscous   mapped_wss     err%
------------------------------------------------
viscous Fx         264.57      -266.99    200.9%
viscous Fy           0.80        -0.97    221.8%
viscous Fz         -10.51        10.57    200.6%

wss magnitude: 2.5478 → 112.5028 Pa
wss mean     : 58.1827 Pa

Dot(wss, normal): -48.0718 → 53.5747
(phải ≈ 0 vì wss tangential to surface)


In [19]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]

# ── Fix 1: flip wss sign ──────────────────────────────────────────
wss_fixed = -wss

# ── Fix 2: project wss onto tangent plane ────────────────────────
# Enlever la composante normale de wss
wss_normal_component = np.sum(wss_fixed * normals, axis=1, keepdims=True)
wss_tangential       = wss_fixed - wss_normal_component * normals

print(f"Before projection - Dot(wss, n): "
      f"{np.sum(wss_fixed*normals, axis=1).mean():.4f}")
print(f"After  projection - Dot(wss, n): "
      f"{np.sum(wss_tangential*normals, axis=1).mean():.6f} (phải ≈ 0)")

# ── Integrate ─────────────────────────────────────────────────────
v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)

def integrate(field):
    fc = (field[tris[:,0]] + field[tris[:,1]] + field[tris[:,2]]) / 3.0
    return np.sum(fc * areas[:, np.newaxis], axis=0)

F_p   = integrate(-p_raw[:, np.newaxis] * normals)
F_wss = integrate(wss_tangential)
F_wing = F_p + F_wss

# ── Tip cap ───────────────────────────────────────────────────────
y_max    = pts[:,1].max()
tip_mask = np.abs(pts[:,1] - y_max) < 1e-4
tip_idx  = np.where(tip_mask)[0]
tip_pts  = pts[tip_idx]
tip_tris = Delaunay(tip_pts[:, [0,2]]).simplices

v0t = tip_pts[tip_tris[:,0]]
v1t = tip_pts[tip_tris[:,1]]
v2t = tip_pts[tip_tris[:,2]]
areas_t    = 0.5 * np.linalg.norm(np.cross(v1t-v0t, v2t-v0t), axis=1)
p_center_t = (p_raw[tip_idx][tip_tris[:,0]] +
              p_raw[tip_idx][tip_tris[:,1]] +
              p_raw[tip_idx][tip_tris[:,2]]) / 3.0
F_tip = np.sum(-p_center_t[:, np.newaxis]
               * np.array([0., 1., 0.])
               * areas_t[:, np.newaxis], axis=0)

F_total = F_wing + F_tip

# ── Validation ────────────────────────────────────────────────────
F_of  = np.array([1133.0760, -4433.4000, 17824.7600])
OF_p  = np.array([868.51,    -4434.20,   17835.28])
OF_wss= np.array([264.57,        0.80,     -10.51])

print(f"\n{'':12} {'OpenFOAM':>12} {'Mapped':>12} {'Error%':>8}")
print(f"{'-'*48}")
for i, l in enumerate(["Fx", "Fy", "Fz"]):
    e = abs(F_p[i] - OF_p[i]) / (abs(OF_p[i]) + 1e-10) * 100
    print(f"{'pressure '+l:12} {OF_p[i]:12.2f} {F_p[i]:12.2f} {e:8.1f}%")
print()
for i, l in enumerate(["Fx", "Fy", "Fz"]):
    e = abs(F_wss[i] - OF_wss[i]) / (abs(OF_wss[i]) + 1e-10) * 100
    print(f"{'viscous '+l:12} {OF_wss[i]:12.2f} {F_wss[i]:12.2f} {e:8.1f}%")
print()
for i, l in enumerate(["Fx", "Fy", "Fz"]):
    e = abs(F_total[i] - F_of[i]) / (abs(F_of[i]) + 1e-10) * 100
    print(f"{'total '+l:12} {F_of[i]:12.2f} {F_total[i]:12.2f} {e:8.1f}%")
err_mag = abs(np.linalg.norm(F_total) - np.linalg.norm(F_of)) / np.linalg.norm(F_of) * 100
print(f"\n|F| error = {err_mag:.2f}%")

Before projection - Dot(wss, n): 0.4632
After  projection - Dot(wss, n): -0.000000 (phải ≈ 0)

                 OpenFOAM       Mapped   Error%
------------------------------------------------
pressure Fx        868.51       451.73     48.0%
pressure Fy      -4434.20     -4293.16      3.2%
pressure Fz      17835.28     17836.00      0.0%

viscous Fx         264.57       267.42      1.1%
viscous Fy           0.80         0.67     16.4%
viscous Fz         -10.51        -9.31     11.4%

total Fx          1133.08       719.15     36.5%
total Fy         -4433.40     -5225.69     17.9%
total Fz         17824.76     17826.69      0.0%

|F| error = 1.02%


In [21]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]
wss_fixed = -wss
wss_n     = np.sum(wss_fixed * normals, axis=1, keepdims=True)
wss_t     = wss_fixed - wss_n * normals

F_of  = np.array([1133.0760, -4433.4000, 17824.7600])

v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
n_center = (normals[tris[:,0]] + normals[tris[:,1]] + normals[tris[:,2]]) / 3.0

def integrate(field):
    fc = (field[tris[:,0]] + field[tris[:,1]] + field[tris[:,2]]) / 3.0
    return np.sum(fc * areas[:, np.newaxis], axis=0)

# Integral of normals → open surface effect
int_n = np.sum(n_center * areas[:, np.newaxis], axis=0)
print(f"Integral normals = {int_n}")
print(f"p_raw_mean * int_n = {p_raw.mean() * int_n}")
print(f"Fx error = {868.51 - 451.73:.2f} N")
print(f"p_inf * nx_int = {54019.55 * int_n[0]:.2f} N")

# Root cap để close surface theo x
# Root cap là airfoil cross-section tại y=0
y_min      = pts[:,1].min()
root_mask  = np.abs(pts[:,1] - y_min) < 1e-4
root_idx   = np.where(root_mask)[0]
root_pts   = pts[root_idx]

# Root cap nằm trên x-z plane → triangulate trên x-z
root_tris  = Delaunay(root_pts[:, [0,2]]).simplices
v0r = root_pts[root_tris[:,0]]
v1r = root_pts[root_tris[:,1]]
v2r = root_pts[root_tris[:,2]]
areas_r    = 0.5 * np.linalg.norm(np.cross(v1r-v0r, v2r-v0r), axis=1)
p_center_r = (p_raw[root_idx][root_tris[:,0]] +
              p_raw[root_idx][root_tris[:,1]] +
              p_raw[root_idx][root_tris[:,2]]) / 3.0
print(f"\nRoot cap area = {areas_r.sum():.6f} m²")

# Root cap normal = (0, -1, 0) → pointing outside domain
# Nhưng root là OPEN → pressure ở root = p_inf (freestream)
# Contribution = -p_inf * n_root * A_root
for sign, label in [(+1, "root n=+y"), (-1, "root n=-y")]:
    F_root = np.sum(-p_center_r[:, np.newaxis]
                    * np.array([0., float(sign), 0.])
                    * areas_r[:, np.newaxis], axis=0)

    # Tip cap
    y_max    = pts[:,1].max()
    tip_mask = np.abs(pts[:,1] - y_max) < 1e-4
    tip_idx  = np.where(tip_mask)[0]
    tip_pts  = pts[tip_idx]
    tip_tris = Delaunay(tip_pts[:, [0,2]]).simplices
    v0t = tip_pts[tip_tris[:,0]]; v1t = tip_pts[tip_tris[:,1]]; v2t = tip_pts[tip_tris[:,2]]
    areas_t    = 0.5 * np.linalg.norm(np.cross(v1t-v0t, v2t-v0t), axis=1)
    p_center_t = (p_raw[tip_idx][tip_tris[:,0]] +
                  p_raw[tip_idx][tip_tris[:,1]] +
                  p_raw[tip_idx][tip_tris[:,2]]) / 3.0
    F_tip = np.sum(-p_center_t[:, np.newaxis]
                   * np.array([0., 1., 0.])
                   * areas_t[:, np.newaxis], axis=0)

    F_wing  = integrate(-p_raw[:, np.newaxis] * normals) + integrate(wss_t)
    F_total = F_wing + F_root + F_tip

    err_mag = abs(np.linalg.norm(F_total) - np.linalg.norm(F_of)) / np.linalg.norm(F_of) * 100
    print(f"\n=== {label} ===")
    for i, l in enumerate(["Fx", "Fy", "Fz"]):
        e = abs(F_total[i] - F_of[i]) / (abs(F_of[i]) + 1e-10) * 100
        print(f"{l}: OF={F_of[i]:10.2f}  mapped={F_total[i]:10.2f}  err={e:.1f}%")
    print(f"|F| error = {err_mag:.2f}%")

Integral normals = [ 0.00771111  0.07531793 -0.00055922]
p_raw_mean * int_n = [ 385.32968  3763.6914    -27.944765]
Fx error = 416.78 N
p_inf * nx_int = 416.55 N

Root cap area = 0.080978 m²

=== root n=+y ===
Fx: OF=   1133.08  mapped=    719.15  err=36.5%
Fy: OF=  -4433.40  mapped=  -9353.14  err=111.0%
Fz: OF=  17824.76  mapped=  17826.69  err=0.0%
|F| error = 9.46%

=== root n=-y ===
Fx: OF=   1133.08  mapped=    719.15  err=36.5%
Fy: OF=  -4433.40  mapped=  -1098.24  err=75.2%
Fz: OF=  17824.76  mapped=  17826.69  err=0.0%
|F| error = 2.87%


In [22]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]
wss_t   = -wss - np.sum(-wss * normals, axis=1, keepdims=True) * normals

F_of = np.array([1133.0760, -4433.4000, 17824.7600])

v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)

def integrate(pts, tris, field):
    fc = (field[tris[:,0]] + field[tris[:,1]] + field[tris[:,2]]) / 3.0
    v0=pts[tris[:,0]]; v1=pts[tris[:,1]]; v2=pts[tris[:,2]]
    a = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
    return np.sum(fc * a[:, np.newaxis], axis=0)

F_wing = integrate(pts, tris, -p_raw[:, np.newaxis] * normals + wss_t)

# ── Root cap: tính normal từ geometry thật ───────────────────────
y_min     = pts[:,1].min()
root_mask = np.abs(pts[:,1] - y_min) < 1e-4
root_idx  = np.where(root_mask)[0]
root_pts  = pts[root_idx]   # airfoil cross-section tại y=0

# Triangulate trên x-z plane
root_tris = Delaunay(root_pts[:, [0,2]]).simplices

# Tính normal thật của root cap (không assume là +/-y)
v0r = root_pts[root_tris[:,0]]
v1r = root_pts[root_tris[:,1]]
v2r = root_pts[root_tris[:,2]]
cross_r  = np.cross(v1r-v0r, v2r-v0r)          # (M, 3)
areas_r  = 0.5 * np.linalg.norm(cross_r, axis=1)
n_root   = cross_r / (2*areas_r[:, np.newaxis])  # unit normals

# Check direction: phải pointing vào trong domain (y > 0)
# Nếu ny < 0 thì flip
if n_root[:,1].mean() < 0:
    n_root = -n_root
    print("Root cap normals flipped to point inward (+y)")
else:
    print("Root cap normals OK (+y direction)")

print(f"Root cap normal mean: nx={n_root[:,0].mean():.4f}, "
      f"ny={n_root[:,1].mean():.4f}, nz={n_root[:,2].mean():.4f}")
print(f"Root cap area: {areas_r.sum():.6f} m²")

# Pressure at root cap
p_center_r = (p_raw[root_idx][root_tris[:,0]] +
              p_raw[root_idx][root_tris[:,1]] +
              p_raw[root_idx][root_tris[:,2]]) / 3.0

# Traction trên root cap (wss=0, open boundary)
traction_r = -p_center_r[:, np.newaxis] * n_root
F_root = np.sum(traction_r * areas_r[:, np.newaxis], axis=0)

print(f"\nRoot cap force:")
print(f"  Fx={F_root[0]:.2f}  Fy={F_root[1]:.2f}  Fz={F_root[2]:.2f}")

# ── Tip cap ───────────────────────────────────────────────────────
y_max    = pts[:,1].max()
tip_mask = np.abs(pts[:,1] - y_max) < 1e-4
tip_idx  = np.where(tip_mask)[0]
tip_pts  = pts[tip_idx]
tip_tris = Delaunay(tip_pts[:, [0,2]]).simplices

v0t = tip_pts[tip_tris[:,0]]
v1t = tip_pts[tip_tris[:,1]]
v2t = tip_pts[tip_tris[:,2]]
cross_t  = np.cross(v1t-v0t, v2t-v0t)
areas_t  = 0.5 * np.linalg.norm(cross_t, axis=1)
n_tip    = cross_t / (2*areas_t[:, np.newaxis])

# Tip pointing outward (+y)
if n_tip[:,1].mean() < 0:
    n_tip = -n_tip
print(f"\nTip cap normal mean: nx={n_tip[:,0].mean():.4f}, "
      f"ny={n_tip[:,1].mean():.4f}, nz={n_tip[:,2].mean():.4f}")

p_center_t = (p_raw[tip_idx][tip_tris[:,0]] +
              p_raw[tip_idx][tip_tris[:,1]] +
              p_raw[tip_idx][tip_tris[:,2]]) / 3.0
traction_t = -p_center_t[:, np.newaxis] * n_tip
F_tip = np.sum(traction_t * areas_t[:, np.newaxis], axis=0)

print(f"Tip cap force:")
print(f"  Fx={F_tip[0]:.2f}  Fy={F_tip[1]:.2f}  Fz={F_tip[2]:.2f}")

# ── Total ─────────────────────────────────────────────────────────
F_total = F_wing + F_root + F_tip

print(f"\n=== Final Validation ===")
print(f"{'':6} {'OpenFOAM':>12} {'Mapped':>12} {'Error%':>8}")
print(f"{'-'*42}")
for i, l in enumerate(["Fx", "Fy", "Fz"]):
    e = abs(F_total[i] - F_of[i]) / (abs(F_of[i]) + 1e-10) * 100
    print(f"{l:6} {F_of[i]:12.2f} {F_total[i]:12.2f} {e:8.1f}%")
err_mag = abs(np.linalg.norm(F_total)-np.linalg.norm(F_of)) / np.linalg.norm(F_of) * 100
print(f"{'|F|':6} {np.linalg.norm(F_of):12.2f} "
      f"{np.linalg.norm(F_total):12.2f} {err_mag:8.2f}%")

Root cap normals flipped to point inward (+y)
Root cap normal mean: nx=0.0000, ny=1.0000, nz=0.0000
Root cap area: 0.080978 m²

Root cap force:
  Fx=0.00  Fy=-4127.45  Fz=0.00

Tip cap normal mean: nx=-0.0001, ny=0.9997, nz=-0.0002
Tip cap force:
  Fx=-0.01  Fy=-933.19  Fz=-0.08

=== Final Validation ===
           OpenFOAM       Mapped   Error%
------------------------------------------
Fx          1133.08       719.02     36.5%
Fy         -4433.40     -9353.16    111.0%
Fz         17824.76     17826.46      0.0%
|F|        18402.74     20144.01     9.46%


In [23]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]
wss_t   = -wss - np.sum(-wss * normals, axis=1, keepdims=True) * normals

F_of    = np.array([1133.0760, -4433.4000, 17824.7600])
OF_p    = np.array([868.51,    -4434.20,   17835.28])
OF_wss  = np.array([264.57,        0.80,     -10.51])

v0    = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
areas = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)

def integrate(field):
    fc = (field[tris[:,0]] + field[tris[:,1]] + field[tris[:,2]]) / 3.0
    return np.sum(fc * areas[:, np.newaxis], axis=0)

# Tip cap only (no root cap - root is open in OpenFOAM)
y_max    = pts[:,1].max()
tip_mask = np.abs(pts[:,1] - y_max) < 1e-4
tip_idx  = np.where(tip_mask)[0]
tip_pts  = pts[tip_idx]
tip_tris = Delaunay(tip_pts[:, [0,2]]).simplices
v0t = tip_pts[tip_tris[:,0]]
v1t = tip_pts[tip_tris[:,1]]
v2t = tip_pts[tip_tris[:,2]]
areas_t    = 0.5 * np.linalg.norm(np.cross(v1t-v0t, v2t-v0t), axis=1)
p_center_t = (p_raw[tip_idx][tip_tris[:,0]] +
              p_raw[tip_idx][tip_tris[:,1]] +
              p_raw[tip_idx][tip_tris[:,2]]) / 3.0

# Test: p_gauge vs p_raw pour wing + tip
print(f"{'pRef':>10} {'Fx':>10} {'Fy':>10} {'Fz':>10} {'|F|err%':>10}")
print("-"*52)

for pRef, label in [(0, "p_raw"), (54019.55, "p_gauge")]:
    p_use = p_raw - pRef

    # Wing
    F_p   = integrate(-p_use[:, np.newaxis] * normals)
    F_wss = integrate(wss_t)
    F_wing = F_p + F_wss

    # Tip cap
    p_tip_use  = p_center_t - pRef
    F_tip = np.sum(-p_tip_use[:, np.newaxis]
                   * np.array([0., 1., 0.])
                   * areas_t[:, np.newaxis], axis=0)

    F_total = F_wing + F_tip
    err = abs(np.linalg.norm(F_total)-np.linalg.norm(F_of))/np.linalg.norm(F_of)*100
    print(f"{label:>10} {F_total[0]:>10.2f} {F_total[1]:>10.2f} "
          f"{F_total[2]:>10.2f} {err:>10.2f}%")
    for i, l in enumerate(["Fx","Fy","Fz"]):
        e = abs(F_total[i]-F_of[i])/(abs(F_of[i])+1e-10)*100
        print(f"  {l}: OF={F_of[i]:9.2f}  mapped={F_total[i]:9.2f}  err={e:.1f}%")
    print()

print(f"OpenFOAM: {F_of}")

      pRef         Fx         Fy         Fz    |F|err%
----------------------------------------------------
     p_raw     719.15   -5225.69   17826.69       1.02%
  Fx: OF=  1133.08  mapped=   719.15  err=36.5%
  Fy: OF= -4433.40  mapped= -5225.69  err=17.9%
  Fz: OF= 17824.76  mapped= 17826.69  err=0.0%

   p_gauge    1135.74     -98.68   17796.29       3.10%
  Fx: OF=  1133.08  mapped=  1135.74  err=0.2%
  Fy: OF= -4433.40  mapped=   -98.68  err=97.8%
  Fz: OF= 17824.76  mapped= 17796.29  err=0.2%

OpenFOAM: [ 1133.076 -4433.4   17824.76 ]


In [24]:
import numpy as np
import meshio
from scipy.spatial import Delaunay

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]
wss_t   = -wss - np.sum(-wss*normals, axis=1, keepdims=True)*normals
F_of    = np.array([1133.0760, -4433.4000, 17824.7600])

# ── Wing surface avec p_gauge ────────────────────────────────────
p_gauge = p_raw - 54019.55
traction = -p_gauge[:, np.newaxis] * normals + wss_t

v0=pts[tris[:,0]]; v1=pts[tris[:,1]]; v2=pts[tris[:,2]]
areas = 0.5*np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)

def integrate(pts, tris, field):
    fc = (field[tris[:,0]]+field[tris[:,1]]+field[tris[:,2]])/3.0
    v0=pts[tris[:,0]]; v1=pts[tris[:,1]]; v2=pts[tris[:,2]]
    a = 0.5*np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
    return np.sum(fc*a[:,np.newaxis], axis=0)

F_wing = integrate(pts, tris, traction)
print(f"F_wing = {F_wing}")

# ── Tip cap avec p_gauge ─────────────────────────────────────────
y_max    = pts[:,1].max()
tip_mask = np.abs(pts[:,1]-y_max) < 1e-4
tip_idx  = np.where(tip_mask)[0]
tip_pts  = pts[tip_idx]
tip_tris = Delaunay(tip_pts[:,[0,2]]).simplices

# Tính normal thật của tip cap từ geometry
v0t=tip_pts[tip_tris[:,0]]; v1t=tip_pts[tip_tris[:,1]]; v2t=tip_pts[tip_tris[:,2]]
cross_t  = np.cross(v1t-v0t, v2t-v0t)
areas_t  = 0.5*np.linalg.norm(cross_t, axis=1)
n_tip    = cross_t / (2*areas_t[:,np.newaxis])
if n_tip[:,1].mean() < 0:
    n_tip = -n_tip

print(f"Tip normal mean: {n_tip.mean(axis=0)}")
print(f"Tip area: {areas_t.sum():.6f} m²")

# p_gauge tại tip
p_tip_g = (p_gauge[tip_idx][tip_tris[:,0]] +
           p_gauge[tip_idx][tip_tris[:,1]] +
           p_gauge[tip_idx][tip_tris[:,2]]) / 3.0

print(f"p_gauge at tip: {p_tip_g.min():.2f} → {p_tip_g.max():.2f} Pa")
print(f"p_gauge mean at tip: {p_tip_g.mean():.2f} Pa")

F_tip = np.sum(-p_tip_g[:,np.newaxis] * n_tip * areas_t[:,np.newaxis], axis=0)
print(f"F_tip = {F_tip}")

F_total = F_wing + F_tip
print(f"\n=== Validation ===")
for i, l in enumerate(["Fx","Fy","Fz"]):
    e = abs(F_total[i]-F_of[i])/(abs(F_of[i])+1e-10)*100
    print(f"{l}: OF={F_of[i]:10.2f}  mapped={F_total[i]:10.2f}  err={e:.1f}%")
err = abs(np.linalg.norm(F_total)-np.linalg.norm(F_of))/np.linalg.norm(F_of)*100
print(f"|F| error = {err:.2f}%")

F_wing = [ 1135.7585   -223.89835 17796.166  ]
Tip normal mean: [-8.5331929e-05  9.9965417e-01 -2.1872943e-04]
Tip area: 0.019593 m²
p_gauge at tip: -27674.83 → -1834.49 Pa
p_gauge mean at tip: -6380.78 Pa
F_tip = [-2.4924186e-04  1.2521186e+02  1.2326589e-02]

=== Validation ===
Fx: OF=   1133.08  mapped=   1135.76  err=0.2%
Fy: OF=  -4433.40  mapped=    -98.69  err=97.8%
Fz: OF=  17824.76  mapped=  17796.18  err=0.2%
|F| error = 3.10%


In [25]:
import numpy as np
import meshio
import pyvista as pv

# Dùng pyvista để extract tip cap đúng cách
mesh_vtp = pv.read("wing.vtp")

# Clip lấy tip region
y_max   = np.array(mesh_vtp.points)[:,1].max()
tip_cap = mesh_vtp.clip('-y', origin=(0, y_max-0.001, 0))

print(f"Tip cap points: {tip_cap.n_points}")
print(f"Tip cap area  : {tip_cap.area:.6f} m²")

# Tính normal của tip cap đúng cách
tip_cap = tip_cap.compute_normals()
tip_normals = np.array(tip_cap.point_data["Normals"])
tip_pts     = np.array(tip_cap.points)
tip_p       = np.array(tip_cap.point_data["p"])

print(f"Tip normal mean: {tip_normals.mean(axis=0)}")
print(f"Tip p range    : {tip_p.min():.2f} → {tip_p.max():.2f} Pa")

# Integrate traction trên tip cap
p_inf      = 54019.55
tip_p_g    = tip_p - p_inf
tip_trac   = -tip_p_g[:, np.newaxis] * tip_normals  # wss=0 trên tip cap

tip_cap_with_trac = pv.PolyData(tip_pts)
tip_cap_with_trac["traction"] = tip_trac
tip_cap_with_trac.save("tip_cap_check.vtp")
print("Saved: tip_cap_check.vtp → check in ParaView")

Tip cap points: 1948
Tip cap area  : 0.020273 m²
Tip normal mean: [ 0.00111387 -0.76482743  0.00329547]
Tip p range    : 25946.17 → 52832.61 Pa
Saved: tip_cap_check.vtp → check in ParaView


In [26]:
import numpy as np
import meshio

msh  = meshio.read("wing_surface.xdmf")
pts  = msh.points
tris = msh.cells_dict["triangle"]

# Tính projected area theo y (spanwise)
v0 = pts[tris[:,0]]; v1 = pts[tris[:,1]]; v2 = pts[tris[:,2]]
cross   = np.cross(v1-v0, v2-v0)
areas   = 0.5 * np.linalg.norm(cross, axis=1)
ny_proj = 0.5 * np.abs(cross[:,1])   # projected area theo y

print(f"Total surface area     : {areas.sum():.4f} m²")
print(f"Total projected area Ay: {ny_proj.sum():.4f} m²")
print(f"Tip cap area           : 0.019593 m²")
print(f"Missing projected area : {ny_proj.sum() - 0.019593:.4f} m²")

# Nếu p_gauge_mean ≈ -4210 / missing_area thì confirm
missing = ny_proj.sum() - 0.019593
print(f"\nRequired p_gauge for missing area: {-4210/missing:.2f} Pa")
print(f"Actual p_gauge mean: {(msh.point_data['p_gauge']+50000-54019.55).mean():.2f} Pa")

Total surface area     : 4.6035 m²
Total projected area Ay: 0.3586 m²
Tip cap area           : 0.019593 m²
Missing projected area : 0.3390 m²

Required p_gauge for missing area: -12417.09 Pa
Actual p_gauge mean: -8068.38 Pa


In [28]:
import numpy as np
import meshio

msh     = meshio.read("wing_surface.xdmf")
pts     = msh.points
tris    = msh.cells_dict["triangle"]
normals = msh.point_data["normals"]
p_raw   = msh.point_data["p_gauge"] + 54019.55
wss     = msh.point_data["shearstress"]

p_inf    = 54019.55
p_gauge  = p_raw - p_inf
wss_t    = -wss - np.sum(-wss*normals, axis=1, keepdims=True)*normals
traction = -p_gauge[:,np.newaxis]*normals + wss_t

v0=pts[tris[:,0]]; v1=pts[tris[:,1]]; v2=pts[tris[:,2]]
cross  = np.cross(v1-v0, v2-v0)
areas  = 0.5*np.linalg.norm(cross, axis=1)
n_cell = cross / (2*areas[:,np.newaxis])  # cell normals

# Tính force per cell
tc = (traction[tris[:,0]]+traction[tris[:,1]]+traction[tris[:,2]])/3.0
F_cell = tc * areas[:,np.newaxis]

# Split upper vs lower surface
cell_centers = (pts[tris[:,0]]+pts[tris[:,1]]+pts[tris[:,2]])/3.0

# Upper surface: nz > 0
upper = n_cell[:,2] > 0
lower = n_cell[:,2] < 0

print(f"Upper surface cells: {upper.sum()}")
print(f"Lower surface cells: {lower.sum()}")
print(f"\nUpper surface force: Fx={F_cell[upper,0].sum():.2f}  "
      f"Fy={F_cell[upper,1].sum():.2f}  Fz={F_cell[upper,2].sum():.2f}")
print(f"Lower surface force: Fx={F_cell[lower,0].sum():.2f}  "
      f"Fy={F_cell[lower,1].sum():.2f}  Fz={F_cell[lower,2].sum():.2f}")
print(f"Total             : Fx={F_cell[:,0].sum():.2f}  "
      f"Fy={F_cell[:,1].sum():.2f}  Fz={F_cell[:,2].sum():.2f}")

# OpenFOAM reference
print(f"\nOpenFOAM: Fx=1133.08  Fy=-4433.40  Fz=17824.76")
print(f"\nConclusion:")
print(f"Fx error: {abs(F_cell[:,0].sum()-1133.08)/1133.08*100:.1f}%")
print(f"Fy error: {abs(F_cell[:,1].sum()-(-4433.40))/4433.40*100:.1f}%")
print(f"Fz error: {abs(F_cell[:,2].sum()-17824.76)/17824.76*100:.1f}%")

Upper surface cells: 190363
Lower surface cells: 186088

Upper surface force: Fx=511.93  Fy=-21.82  Fz=19077.37
Lower surface force: Fx=623.20  Fy=-279.18  Fz=-1282.07
Total             : Fx=1135.75  Fy=-223.90  Fz=17796.33

OpenFOAM: Fx=1133.08  Fy=-4433.40  Fz=17824.76

Conclusion:
Fx error: 0.2%
Fy error: 94.9%
Fz error: 0.2%


In [30]:
# VERIFICATION WHETHER TIPCAP CONTRIBUTES TO ERROR IN WINGSPAN FORCE (FY)

import numpy as np
import pyvista as pv

# ── Load OpenFOAM VTP ────────────────────────────────────────────
mesh_vtp = pv.read("wing.vtp")
pts_of   = np.array(mesh_vtp.points)
p_of     = np.array(mesh_vtp.point_data["p"])
wss_of   = np.array(mesh_vtp.point_data["wallShearStress"])

# Tính normals - dùng vtk trực tiếp như trước (đã proven working)
import vtk
from vtk.util.numpy_support import vtk_to_numpy

data_reader = vtk.vtkXMLPolyDataReader()
data_reader.SetFileName("wing.vtp")
data_reader.Update()
poly = data_reader.GetOutput()

triangulate = vtk.vtkTriangleFilter()
triangulate.SetInputData(poly)
triangulate.Update()
poly = triangulate.GetOutput()

normal_filter = vtk.vtkPolyDataNormals()
normal_filter.SetInputData(poly)
normal_filter.ComputePointNormalsOn()
normal_filter.ComputeCellNormalsOff()
normal_filter.AutoOrientNormalsOn()
normal_filter.ConsistencyOn()
normal_filter.SplittingOff()
normal_filter.Update()
poly = normal_filter.GetOutput()

pts_of = vtk_to_numpy(poly.GetPoints().GetData())
p_of   = vtk_to_numpy(poly.GetPointData().GetArray("p"))
wss_of = vtk_to_numpy(poly.GetPointData().GetArray("wallShearStress"))
n_of   = vtk_to_numpy(poly.GetPointData().GetArray("Normals"))

# Extract triangles
cells = poly.GetPolys()
cells.InitTraversal()
idList = vtk.vtkIdList()
tris = []
while cells.GetNextCell(idList):
    tris.append([idList.GetId(i) for i in range(3)])
tris = np.array(tris)

print(f"Points    : {pts_of.shape}")
print(f"Triangles : {tris.shape}")
print(f"y range   : {pts_of[:,1].min():.4f} → {pts_of[:,1].max():.4f}")

# ── Traction ─────────────────────────────────────────────────────
p_inf   = 54019.55
p_gauge = p_of - p_inf
wss_t   = -wss_of - np.sum(-wss_of*n_of, axis=1, keepdims=True)*n_of
traction = -p_gauge[:,None]*n_of + wss_t

# ── Classify triangles: tip cap vs wing skin ──────────────────────
y_max = pts_of[:,1].max()
tri_cy = (pts_of[tris[:,0],1] + pts_of[tris[:,1],1] + pts_of[tris[:,2],1]) / 3.0

tip_tri_mask  = tri_cy > (y_max - 0.01)
skin_tri_mask = ~tip_tri_mask

print(f"Tip cap triangles : {tip_tri_mask.sum()}")
print(f"Wing skin triangles: {skin_tri_mask.sum()}")

# ── Integrate ────────────────────────────────────────────────────
def integrate_force(tris_sel):
    v0 = pts_of[tris_sel[:,0]]
    v1 = pts_of[tris_sel[:,1]]
    v2 = pts_of[tris_sel[:,2]]
    areas    = 0.5 * np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
    t_center = (traction[tris_sel[:,0]] +
                traction[tris_sel[:,1]] +
                traction[tris_sel[:,2]]) / 3.0
    return np.sum(t_center * areas[:,None], axis=0)

F_skin  = integrate_force(tris[skin_tri_mask])
F_tip   = integrate_force(tris[tip_tri_mask])
F_total = F_skin + F_tip
F_of    = np.array([1133.08, -4433.40, 17824.76])

print(f"\n=== Force Decomposition ===")
print(f"{'':12} {'Fx':>10} {'Fy':>10} {'Fz':>10}")
print(f"{'-'*44}")
print(f"{'Wing skin':12} {F_skin[0]:10.2f} {F_skin[1]:10.2f} {F_skin[2]:10.2f}")
print(f"{'Tip cap':12} {F_tip[0]:10.2f}  {F_tip[1]:10.2f} {F_tip[2]:10.2f}")
print(f"{'Total':12} {F_total[0]:10.2f} {F_total[1]:10.2f} {F_total[2]:10.2f}")
print(f"{'OpenFOAM':12} {F_of[0]:10.2f} {F_of[1]:10.2f} {F_of[2]:10.2f}")

print(f"\n=== Error after tip cap ===")
for i, l in enumerate(["Fx","Fy","Fz"]):
    e = abs(F_total[i]-F_of[i])/(abs(F_of[i])+1e-10)*100
    print(f"{l}: {e:.1f}%")

Points    : (189555, 3)
Triangles : (378479, 3)
y range   : 0.0000 → 3.0001
Tip cap triangles : 3745
Wing skin triangles: 374734

=== Force Decomposition ===
                     Fx         Fy         Fz
--------------------------------------------
Wing skin       1135.01    -359.95   17768.85
Tip cap            0.75      136.05      27.34
Total           1135.75    -223.90   17796.19
OpenFOAM        1133.08   -4433.40   17824.76

=== Error after tip cap ===
Fx: 0.2%
Fy: 94.9%
Fz: 0.2%


In [31]:
import numpy as np

# Dùng lại data từ trên
F_of = np.array([1133.08, -4433.40, 17824.76])

# Case 1: p_gauge (current approach)
trac_gauge = -p_gauge[:,None]*n_of + wss_t

# Case 2: absolute pressure pRef=0 (như OpenFOAM)
trac_abs   = -p_of[:,None]*n_of + wss_t

def integrate_all(traction):
    v0=pts_of[tris[:,0]]; v1=pts_of[tris[:,1]]; v2=pts_of[tris[:,2]]
    areas = 0.5*np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
    tc = (traction[tris[:,0]]+traction[tris[:,1]]+traction[tris[:,2]])/3.0
    return np.sum(tc*areas[:,None], axis=0)

F_gauge = integrate_all(trac_gauge)
F_abs   = integrate_all(trac_abs)

# Integral of normals
v0=pts_of[tris[:,0]]; v1=pts_of[tris[:,1]]; v2=pts_of[tris[:,2]]
areas = 0.5*np.linalg.norm(np.cross(v1-v0, v2-v0), axis=1)
nc = (n_of[tris[:,0]]+n_of[tris[:,1]]+n_of[tris[:,2]])/3.0
int_n = np.sum(nc*areas[:,None], axis=0)

print(f"Integral of normals: {int_n}")
print(f"p_inf * int_n      : {p_inf * int_n}")
print(f"\nF_gauge = {F_gauge}")
print(f"F_abs   = {F_abs}")
print(f"F_of    = {F_of}")
print(f"\nF_abs - F_gauge = {F_abs - F_gauge}")
print(f"p_inf * int_n   = {p_inf * int_n}  ← phải match line trên")
print(f"\nConclusion:")
print(f"F_abs_y  = {F_abs[1]:.2f} N  vs  OF_y = {F_of[1]:.2f} N")
print(f"Remaining Fy error = {abs(F_abs[1]-F_of[1]):.2f} N")

Integral of normals: [ 0.00771111  0.07531793 -0.00055922]
p_inf * int_n      : [ 416.5506  4068.6406   -30.20896]

F_gauge = [ 1135.7585   -223.89835 17796.166  ]
F_abs   = [  719.0342 -4292.5244 17826.535 ]
F_of    = [ 1133.08 -4433.4  17824.76]

F_abs - F_gauge = [ -416.72437 -4068.626      30.36914]
p_inf * int_n   = [ 416.5506  4068.6406   -30.20896]  ← phải match line trên

Conclusion:
F_abs_y  = -4292.52 N  vs  OF_y = -4433.40 N
Remaining Fy error = 140.88 N
